# this is training the CNNPZ model on the noisy mock data with randomly dropping bands

In [1]:
import sys

from packaging import version
import sklearn
from sklearn.model_selection import KFold, train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

#import tensorflow_probability as tfp

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import pandas as pd
import h5py
import numpy as np

import json
import os

from matplotlib import colors

/home/jaimerzp/anaconda3/envs/qp/lib/python3.13/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_array, issparse
I0000 00:00:1789573576.583652  124200 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789573576.584047  124200 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789573576.619105  124200 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789573577.477659  

In [2]:
import cnnpz

In [3]:
# Parametric paths (edit via environment variables, or defaults below)
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")
PROJECT_HOME = os.environ.get("CNNPZ_PROJECT_HOME", f"/global/homes/{USER[0]}/{USER}/UCL")

# Local data folder (train/test parquet files + filter curves live here together)
LOCAL_DATA_ROOT = os.environ.get("CNNPZ_LOCAL_DATA_ROOT", os.path.join(os.getcwd(), "data"))

CARDINAL_DATA_ROOT = LOCAL_DATA_ROOT + "/"
MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
PRETRAINING_DATA_ROOT = os.path.join(PSCRATCH, "pop-cosmos-data")
FILTER_ROOT = LOCAL_DATA_ROOT + "/"


## Load training and test data, filter curves

In [4]:
saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "train_100k_noisy_y1_i23.parquet"
training_y1 = pd.read_parquet(fname)
training_y1 = training_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "train_100k_noisy_y10_i25.4.parquet"
training_y10 = pd.read_parquet(fname)
training_y10 = training_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "test_100k_noisy_y1_i23.parquet"
test_y1 = pd.read_parquet(fname)
test_y1 = test_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "test_100k_noisy_y10_i25.4.parquet"
test_y10 = pd.read_parquet(fname)
test_y10 = test_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

### Transform data, make X_train, Y_train, and test dataset, variants of this dataset

In [ ]:
# get the LSST and roman filter curves:
filter_root = FILTER_ROOT

wave = {
    "Y":106,
    "J":129,
    "H":158,
}
lsst_filter_curves = {}
roman_filter_curves = {}
for b in "ugrizy":
  lsst_filter_curves[b] = np.loadtxt(filter_root + f'DC2LSST_{b}.res')

for b in "YJH":
  roman_filter_curves[b] = np.loadtxt(filter_root + f'roman_{b}{wave[b]}.res')

lambda_min = lsst_filter_curves['u'][:,0].min()
lambda_max = roman_filter_curves['H'][:,0].max()
print(lambda_min, lambda_max)

## Continuous filter-curve representation

Each band's real transmission curve is interpolated onto a shared wavelength grid and combined into a "filter bank" that fairly splits credit between overlapping filters (no double-counting), then a galaxy's photometry is turned into a continuous curve over wavelength and binned to 32 points to match the CNN's input length.

$$
T_b(\lambda) = \text{interpolated raw transmission curve of band } b \text{ on the common grid}
$$

$$
\text{ownership}_b(\lambda) = \frac{T_b(\lambda)}{\sum_{b'} T_{b'}(\lambda)}, \qquad \sum_b \text{ownership}_b(\lambda) = 1 \ \ \text{wherever} \ \sum_{b'} T_{b'}(\lambda) > 0
$$

$$
\text{shape}_b(\lambda) = \frac{T_b(\lambda)}{\max_\lambda T_b(\lambda)}
$$

$$
\text{filters\_array}_b(\lambda) = \text{ownership}_b(\lambda)\,\text{shape}_b(\lambda)
$$

And the coverage channel, built from ownership alone, for a galaxy with per-band observed flags $o_b \in \{0,1\}$:

$$
\text{coverage}(\lambda) = \sum_b o_b \cdot \text{ownership}_b(\lambda) \ \in [0,1]
$$

In [ ]:
# build the filter bank (avoids double-counting overlapping filters) and bin it
# down to the CNN's 32 input bins
lambda_common = np.linspace(lambda_min, lambda_max, 1000)
bands = "ugrizyJH"
filter_curves = {**lsst_filter_curves, **roman_filter_curves}
filters_array, ownership = cnnpz.build_filter_bank(filter_curves, bands, lambda_common)

n_lambda_bins = 32
lambda_bin_centers, bin_idx = cnnpz.make_lambda_bins(lambda_common, n_lambda_bins)
filters_binned = cnnpz.build_binned_filter_operator(filters_array, bin_idx, n_lambda_bins)
ownership_binned = cnnpz.build_binned_filter_operator(ownership, bin_idx, n_lambda_bins)
print("filters_array:", filters_array.shape, "filters_binned:", filters_binned.shape)

In [ ]:
# ensure a clean positional index before building datasets -- train_ensembles indexes
# Y positionally via KFold, which silently breaks on a non-default index
training_y1 = training_y1.reset_index(drop=True)
test_y1 = test_y1.reset_index(drop=True)

X_y1, Y_y1 = cnnpz.transform_data_to_XY(
    training_y1, filters_binned, ownership_binned, bin_idx, lambda_bin_centers, bands, apply_stretch=False)
X_test_y1, Y_test_y1 = cnnpz.transform_data_to_XY(
    test_y1, filters_binned, ownership_binned, bin_idx, lambda_bin_centers, bands, apply_stretch=False)
print(X_y1.shape, Y_y1.shape)

## Visualize and train the model (Y1)

In [ ]:
cnnpz.visualize_the_data(X_y1, Y_y1, lambda_bin_centers, title="Y1 example")

In [ ]:
# train on Y1 complete
from cnnpz import build_model
trained_models, histories = cnnpz.train_ensembles(build_model, X_y1, Y_y1)

In [ ]:
cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.001))

In [ ]:
# save the trained model:
save_dir = "./models/y1_ensemble_CNN_6layers"
cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
# Final prediction on test set
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(
    Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=False,
    redshift_bins=redshift_bins, imag_bins=imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(),
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')